# Notebook 4 — Visualization

Reads CSV outputs from Notebook 3 and produces 2D ligand images, 3D viewers,
an affinity chart, an interaction heatmap, a summary JSON, and an HTML gallery.

All figures are prioritization aids only.

In [1]:
# CELL 1 — Config and paths
import sys, json, shutil
from pathlib import Path
from datetime import datetime

PROJECT_NAME = "DDR1_Workflow"
try:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
except ImportError:
    pass
_search = [Path("/content/drive/MyDrive") / PROJECT_NAME, Path("/content"),
           Path.home(), Path.cwd(), *Path.cwd().parents]
for _c in _search:
    if (_c / "ddr1_config.py").exists():
        sys.path.insert(0, str(_c)); break
else:
    raise FileNotFoundError("ddr1_config.py not found.")
sys.modules.pop("ddr1_config", None)

from ddr1_config import setup_paths, CONFIG

PATHS = setup_paths(project_name=PROJECT_NAME)
MANUSCRIPT = PATHS["CANDIDATE_MANUSCRIPT"]
ANALYSIS_DIR = PATHS["CANDIDATE_ANALYSIS"]
PLIP_DIR = PATHS["CANDIDATE_PLIP"]
COMPLEX_DIR = PATHS["CANDIDATE_COMPLEXES"]
LIGAND_PREP = PATHS["CANDIDATE_LIGAND_PREP"]

VIS_DIR = PATHS["VISUALIZATION"]
FIG_DIR = PATHS["VIS_FIGURES"]
FIG3D_DIR = PATHS["VIS_3D"]
REPORT_DIR = PATHS["VIS_REPORTS"]
for d in (FIG_DIR, FIG3D_DIR, REPORT_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("VIS_DIR:", VIS_DIR)

Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Environment : COLAB
Project Root: /content/drive/MyDrive/DDR1_Workflow
Directories : 30
VIS_DIR: /content/drive/MyDrive/DDR1_Workflow/visualization


In [2]:
# CELL 2 — Imports
%pip install rdkit
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from rdkit import Chem, RDLogger
from rdkit.Chem import Draw
import plotly.express as px
RDLogger.DisableLog("rdApp.*")
print("Imports OK")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.8/31.8 MB 27.9 MB/s eta 0:00:0000:0100:01m
Imports OK


In [3]:
# CELL 3 — Load NB3 outputs
INTEGRATED_CSV = ANALYSIS_DIR / "integrated_final_analysis.csv"
PLIP_SUMMARY_CSV = PLIP_DIR / "plip_interactions_summary.csv"
REPRESENTATIVE_CSV = PATHS["CANDIDATE_REPRESENTATIVE"] / "representative_pose_selection.csv"
PREPARED_CSV = LIGAND_PREP / "prepared_candidates.csv"

df_results = pd.read_csv(INTEGRATED_CSV) if INTEGRATED_CSV.exists() else None
plip_summary = pd.read_csv(PLIP_SUMMARY_CSV) if PLIP_SUMMARY_CSV.exists() else None
representative = pd.read_csv(REPRESENTATIVE_CSV) if REPRESENTATIVE_CSV.exists() else None
prepared = pd.read_csv(PREPARED_CSV) if PREPARED_CSV.exists() else None

if df_results is None:
    raise FileNotFoundError(f"{INTEGRATED_CSV} not found. Run Notebook 3 first.")
print(f"Integrated results: {len(df_results)} rows")
print(f"PLIP summary: {0 if plip_summary is None else len(plip_summary)} rows")
print(f"Representative: {0 if representative is None else len(representative)} rows")
print(f"Prepared: {0 if prepared is None else len(prepared)} rows")

Integrated results: 6 rows
PLIP summary: 6 rows
Representative: 6 rows
Prepared: 6 rows


In [4]:
# CELL 4 — 2D ligand images
lig2d_dir = FIG_DIR / "ligand_2d"; lig2d_dir.mkdir(parents=True, exist_ok=True)
def norm(v): return "".join(c.lower() for c in str(v) if c.isalnum())

if prepared is None:
    print("No prepared_candidates.csv — skipping 2D.")
else:
    smi_col = next((c for c in ("Canonical_SMILES", "SMILES", "canonical_smiles") if c in prepared.columns), None)
    name_col = next((c for c in ("Name", "compound", "Compound") if c in prepared.columns), None)
    if not smi_col or not name_col:
        print(f"Missing name/SMILES in prepared. Columns: {list(prepared.columns)}")
    else:
        by_name = {norm(r[name_col]): r[smi_col] for _, r in prepared.iterrows()
                   if pd.notna(r[name_col]) and pd.notna(r[smi_col])}
        def find(name):
            k = norm(name)
            if k in by_name: return by_name[k]
            hits = [s for n, s in by_name.items() if k.endswith(n) or n.endswith(k)]
            return hits[0] if len(hits) == 1 else None
        gen = 0
        for _, row in df_results.iterrows():
            name = str(row["Name"]); smi = find(name)
            mol = Chem.MolFromSmiles(smi) if smi else None
            if mol is None: continue
            safe = "".join(c if c.isalnum() or c in "_-" else "_" for c in name).strip("_")
            Draw.MolToFile(mol, str(lig2d_dir / f"{safe}.png"), size=(600, 450), legend=name)
            gen += 1
        print(f"Generated {gen} 2D images in {lig2d_dir}")

Generated 6 2D images in /content/drive/MyDrive/DDR1_Workflow/visualization/figures/ligand_2d


In [5]:
# CELL 5 — 3D viewers
%pip install py3Dmol
cfiles = sorted(COMPLEX_DIR.glob("*_complex.pdb")) if COMPLEX_DIR.exists() else []
if not cfiles:
    print(f"No complexes in {COMPLEX_DIR}")
else:
    try:
        import py3Dmol
    except ImportError:
        print("py3Dmol not installed — skipping 3D.")
    else:
        written = []
        for c in cfiles:
            v = py3Dmol.view(width=900, height=650)
            v.addModel(c.read_text(errors="replace"), "pdb")
            v.setStyle({"cartoon": {"color": "spectrum"}})
            v.addStyle({"hetflag": True}, {"stick": {"radius": 0.22, "colorscheme": "greenCarbon"}})
            v.setBackgroundColor("white"); v.zoomTo()
            out = FIG3D_DIR / f"{c.stem}.html"
            v.write_html(str(out)); written.append(out)
        print(f"Wrote {len(written)} 3D viewers to {FIG3D_DIR}")

Wrote 6 3D viewers to /content/drive/MyDrive/DDR1_Workflow/visualization/3D


In [6]:
# CELL 6 — Affinity chart (PNG + plotly HTML)
affinity_dir = FIG_DIR / "affinities"; affinity_dir.mkdir(parents=True, exist_ok=True)
aff_col = next((c for c in df_results.columns if "Best_affinity" in c), None)
tier_col = "Priority_tier" if "Priority_tier" in df_results.columns else None
if aff_col is None:
    print(f"No affinity column. Columns: {list(df_results.columns)}")
else:
    d = df_results[["Name", aff_col] + ([tier_col] if tier_col else [])].dropna(subset=[aff_col])
    d = d.sort_values(aff_col, ascending=True).reset_index(drop=True)
    tc = {"Tier-A": "#1a9850", "Tier-B": "#f0c000", "Tier-C": "#cccccc",
          "Tier-C (pose unstable)": "#e07a5f", "Tier-C (no score)": "#aaaaaa"}
    colors = [tc.get(str(t), "#cccccc") for t in d[tier_col]] if tier_col else "#4c72b0"
    fig, ax = plt.subplots(figsize=(max(8, 0.55 * len(d)), 6))
    ax.barh(d["Name"], d[aff_col], color=colors, edgecolor="black", linewidth=0.4)
    ax.set_xlabel("Vina best score (kcal/mol)"); ax.set_ylabel("")
    ax.set_title("DDR1 candidates — Vina best score")
    ax.grid(axis="x", linestyle=":", alpha=0.4)
    plt.tight_layout()
    plt.savefig(affinity_dir / "binding_affinities.png", dpi=300, bbox_inches="tight")
    plt.close()
    fig_px = px.bar(d, x=aff_col, y="Name", orientation="h",
                    color=tier_col if tier_col else None,
                    color_discrete_map=tc if tier_col else None,
                    title="DDR1 candidates — Vina best score (kcal/mol)",
                    labels={aff_col: "Vina score (kcal/mol)"})
    fig_px.update_layout(yaxis={"categoryorder": "total ascending"})
    fig_px.write_html(str(affinity_dir / "binding_affinities.html"))
    print(f"Affinity figures written to {affinity_dir}")

Affinity figures written to /content/drive/MyDrive/DDR1_Workflow/visualization/figures/affinities


In [7]:
# CELL 7 — Interaction heatmap
interaction_dir = FIG_DIR / "interactions"; interaction_dir.mkdir(parents=True, exist_ok=True)
ITYPES = ["H-bond", "Hydrophobic", "Salt bridge", "Pi-stack", "Pi-cation", "Halogen", "Water bridge"]
if plip_summary is None or plip_summary.empty:
    print("No PLIP summary — skipping heatmap.")
else:
    present = [t for t in ITYPES if t in plip_summary.columns]
    if not present or "Name" not in plip_summary.columns:
        print(f"PLIP summary missing expected columns: {list(plip_summary.columns)}")
    else:
        heat = plip_summary.set_index("Name")[present].apply(pd.to_numeric, errors="coerce").fillna(0)
        plt.figure(figsize=(max(8, len(present) * 1.15), max(4, len(heat) * 0.42)))
        sns.heatmap(heat, annot=True, fmt=".0f", cmap="YlGnBu", linewidths=0.5)
        plt.title("PLIP interaction counts by compound")
        plt.xlabel("Interaction type"); plt.ylabel("Compound")
        plt.tight_layout()
        plt.savefig(interaction_dir / "interaction_heatmap.png", dpi=300, bbox_inches="tight")
        plt.close()
        fig_px = px.imshow(heat, text_auto=True, aspect="auto", color_continuous_scale="YlGnBu",
                           labels={"x": "Interaction type", "y": "Compound", "color": "Count"},
                           title="PLIP interaction counts by compound")
        fig_px.write_html(str(interaction_dir / "interaction_heatmap.html"))
        print(f"Heatmap written to {interaction_dir}")

Heatmap written to /content/drive/MyDrive/DDR1_Workflow/visualization/figures/interactions


In [ ]:
# CELL 8 — Summary JSON
summary = {"report_date": datetime.now().isoformat(),
           "total_compounds": int(len(df_results)),
           "visualization_dir": str(VIS_DIR),
           "figures_dir": str(FIG_DIR), "figures_3d_dir": str(FIG3D_DIR),
           "report_dir": str(REPORT_DIR),
           "interpretation": ("Vina scores and PLIP contacts are used for computational "
                              "prioritization only. Not Delta-G values.")}
if "Priority_tier" in df_results.columns:
    summary["priority_tier_counts"] = df_results["Priority_tier"].value_counts().to_dict()
if aff_col is not None:
    a = pd.to_numeric(df_results[aff_col], errors="coerce").dropna()
    if len(a):
        summary.update({"best_affinity_kcal_mol": float(a.min()),
                        "worst_affinity_kcal_mol": float(a.max()),
                        "mean_affinity_kcal_mol": float(a.mean())})
with open(REPORT_DIR / "visualization_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(f"Summary: {REPORT_DIR / 'visualization_summary.json'}")

Summary: /content/drive/MyDrive/DDR1_Workflow/visualization/reports/visualization_summary.json


In [9]:
# CELL 9 — HTML gallery
def rel(p): return Path(p).relative_to(VIS_DIR).as_posix()
SECTIONS = [("Binding affinities", "figures/affinities"),
            ("Protein-ligand interactions", "figures/interactions"),
            ("2D ligand structures", "figures/ligand_2d"),
            ("3D complex viewers", "3D")]
SUFFIXES = {".html", ".png", ".svg", ".pdf", ".jpg", ".jpeg"}
sections = []
for title, sub in SECTIONS:
    d = VIS_DIR / sub
    if not d.exists(): continue
    entries = [f'<li><a href="{rel(p)}">{p.name}</a></li>'
               for p in sorted(d.iterdir()) if p.is_file() and p.suffix.lower() in SUFFIXES]
    if entries:
        sections.append(f"<h2>{title}</h2><ul>" + "".join(entries) + "</ul>")
body = "\n".join(sections) if sections else "<p>No figures found.</p>"
ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
env = "COLAB" if CONFIG.is_colab else "LOCAL"
html = f"""<!DOCTYPE html><html><head><meta charset="utf-8">
<title>DDR1 Docking Analysis</title>
<style>body{{font-family:Arial,sans-serif;margin:0;padding:20px;background:#f5f5f5;color:#222}}
.c{{max-width:1100px;margin:0 auto;background:#fff;padding:24px 32px;border-radius:8px;box-shadow:0 2px 6px rgba(0,0,0,.08)}}
h1{{margin-top:0}} h2{{margin-top:28px;border-bottom:1px solid #eee;padding-bottom:4px}}
ul{{list-style:none;padding-left:0}} li{{padding:4px 0}}
a{{color:#0066cc;text-decoration:none}} a:hover{{text-decoration:underline}}
.meta{{color:#666;font-size:.9em}}
.note{{background:#fffbe6;border-left:4px solid #f0c000;padding:12px 16px;margin:20px 0;font-size:.95em}}
</style></head><body><div class="c">
<h1>DDR1 Docking Analysis</h1>
<p class="meta"><strong>Generated:</strong> {ts} | <strong>Environment:</strong> {env}</p>
<div class="note">Vina scores and PLIP contacts are prioritization aids only.
They are not &Delta;G values and were not validated against DDR1 actives, decoys, or
experimental binding data in this study.</div>
{body}
</div></body></html>"""
gallery = VIS_DIR / "index.html"
gallery.write_text(html, encoding="utf-8")
print(f"Gallery: {gallery}")
print(f"Sections: {len(sections)}")

Gallery: /content/drive/MyDrive/DDR1_Workflow/visualization/index.html
Sections: 4


In [ ]:
# CELL 10 — Final summary
print("=" * 70)
print("VISUALIZATION COMPLETE")
print(f"Figures    : {FIG_DIR}")
print(f"3D         : {FIG3D_DIR}")
print(f"Reports    : {REPORT_DIR}")
print(f"Gallery    : {gallery}")
print("\nAll four notebooks complete.")

VISUALIZATION COMPLETE
Figures    : /content/drive/MyDrive/DDR1_Workflow/visualization/figures
3D         : /content/drive/MyDrive/DDR1_Workflow/visualization/3D
Reports    : /content/drive/MyDrive/DDR1_Workflow/visualization/reports
Gallery    : /content/drive/MyDrive/DDR1_Workflow/visualization/index.html

All four notebooks complete.
